In [ ]:
   import os
from pathlib import Path

from dotenv import load_dotenv

env_path = Path("/home/bruno/Projects/Python/etl-openweather-api/config/.env")
load_dotenv(env_path)

print("API_KEY carregada:", bool(os.getenv("API_KEY")))


In [5]:
from __future__ import annotations

import json
import os
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# Define a raiz do projeto de forma absoluta (funciona em scripts e consoles interativos)
# Se estiver executando dentro da pasta src/ ou notebooks/, ajuste conforme necessário:
project_root = Path("/home/bruno/Projects/Python/etl-openweather-api")

env_path = project_root / "config" / ".env"
load_dotenv(env_path)

# Substitui o __file__ pela raiz absoluta do projeto
path_save_dataframe = project_root / "data" / "validated"
path_name = project_root / "data" / "raw" / "weather_data.json"

columns_name_to_drop = ["weather", "weather_icon"]
columns_name_to_rename = {
    "base": "base",
    "visibility": "visibility",
    "dt": "datetime",
    "timezone": "timezone",
    "id": "city_id",
    "name": "city_name",
    "cod": "code",
    "coord.lon": "longitude",
    "coord.lat": "latitude",
    "main.temp": "temperatura",
    "main.feels_like": "feels_like",
    "main.temp_min": "temp_min",
    "main.temp_max": "temp_max",
    "main_pressure": "pressure",
    "main.humidity": "humidity",
    "main.sea_level": "sea_level",
    "main.grnd_level": "grnd_level",
    "wind.speed": "wind_speed",
    "wind.deg": "wind_deg",
    "wind.gust": "wind_gust",
    "clouds.all": "clouds",
    "sys.type": "sys.type",
    "sys.id": "sys_id",
    "sys_country": "country",
    "sys.sunrise": "sunrise",
    "sys.sunset": "sunset"
}
columns_normalize_datetime = ["datetime", "sunrise", "sunset"]

In [10]:
def create_dataframe(path_name: str | Path) -> pd.DataFrame:
    print("Criando o DataFrame com base no json")

    path = Path(path_name)

    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.json_normalize(data)
    print("DataFrame criado com sucesso!")

    return df

def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    print("Inicializando a normalização")
    df_weather = pd.json_normalize(df['weather'].apply(lambda x: x[0]))

    df_weather = df_weather.rename(columns={
        "id": "weather_id",
        "main": "weather_main",
        "desciption": "weather_description",
        "icon": "weather_icon"
    })

    df = pd.concat([df,df_weather], axis=1)
    print("DataFrame normalizado")

    return df

def drop_columns(df: pd.DataFrame, columns_name: list[str]) -> pd.DataFrame:
    print(f"Removendo colunas: {columns_name}")
    
    df = df.drop(columns=columns_name)

    print("Colunas removidas")
    return df

def rename_columns(df: pd.DataFrame, columns_name: dict[str,str]) -> pd.DataFrame:
    print(f"Renomeando as colunas: {columns_name}")
    df = df.rename(columns=columns_name)
    print("Renomeada as colunas")
    return df

def normalize_datetime_column(df: pd.DataFrame, columns_name: list[str]) -> pd.DataFrame:
    print(f"Convertendo as colunas para Datetime: {columns_name}")
    for name in columns_name:
        tz = os.getenv("UTC") or "UTC"
        df[name] = (
            pd.to_datetime(df[name], unit="s", utc=True).dt.tz_convert(tz)
        )
    print("Colunas convertidas para Datetime")
    return df

def save_dataframe(df: pd.DataFrame, path_save_dataframe: str | Path) -> None: 
    print("Salvando o arquivo em formato parquet")
    path = Path(path_save_dataframe)
    path.mkdir(parents=True, exist_ok=True)
    file_path = path / "weather.parquet"
    df.to_parquet(file_path)
    print("Arquivo salvo com sucesso")


def data_transformation() -> None:
    print("Inicializando a transformação")

    df = create_dataframe(path_name)
    print("Passo 1 ok: create_dataframe")

    df = normalize_dataframe(df)
    print("Passo 2 ok: normalize_dataframe")

    df = drop_columns(df, columns_name_to_drop)
    print("Passo 3 ok: drop_columns")

    df = rename_columns(df, columns_name_to_rename)
    print("Passo 4 ok: rename_columns")

    df = normalize_datetime_column(df, columns_normalize_datetime)
    print("Passo 5 ok: normalize_datetime_column")
    df.info()
    #save_dataframe(df, path_save_dataframe)
    print("Passo 6 ok: save_dataframe")

    print("Transformação concluída com sucesso!")

data_transformation()

Inicializando a transformação
Criando o DataFrame com base no json
DataFrame criado com sucesso!
Passo 1 ok: create_dataframe
Inicializando a normalização
DataFrame normalizado
Passo 2 ok: normalize_dataframe
Removendo colunas: ['weather', 'weather_icon']
Colunas removidas
Passo 3 ok: drop_columns
Renomeando as colunas: {'base': 'base', 'visibility': 'visibility', 'dt': 'datetime', 'timezone': 'timezone', 'id': 'city_id', 'name': 'city_name', 'cod': 'code', 'coord.lon': 'longitude', 'coord.lat': 'latitude', 'main.temp': 'temperatura', 'main.feels_like': 'feels_like', 'main.temp_min': 'temp_min', 'main.temp_max': 'temp_max', 'main_pressure': 'pressure', 'main.humidity': 'humidity', 'main.sea_level': 'sea_level', 'main.grnd_level': 'grnd_level', 'wind.speed': 'wind_speed', 'wind.deg': 'wind_deg', 'wind.gust': 'wind_gust', 'clouds.all': 'clouds', 'sys.type': 'sys.type', 'sys.id': 'sys_id', 'sys_country': 'country', 'sys.sunrise': 'sunrise', 'sys.sunset': 'sunset'}
Renomeada as colunas
Pas